# MalWeave RanDS Audit

Run the native/.NET audit on CPU in a private notebook with Internet enabled. Attach
`AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` as Kaggle Secrets. For a temporary
AWS session, attach `AWS_SESSION_TOKEN` too. The session needs read access to the
private input bundle and the original RanDS files.

Set `LIMIT = 1000` to attempt at most 1,000 sources in this session. The default
`LIMIT = None` processes the remaining inventory. The session reserves time to
save a SQLite checkpoint before Kaggle's time limit.

Use **Save Version → Save & Run All**. The notebook's Output contains
`rands-audit/state.sqlite3`, `summary.json` and `console.log`.
The final `native-dotnet.csv` is saved after the inventory is resolved. Attach
a previous Output to resume. Set `RESUME_DB` to select a specific checkpoint.

The staged RanDS bundle includes the verified local checkpoint. Attach a newer checkpoint to continue from later work.


In [ ]:
DATASET = "rands"
LIMIT = None
import os
from pathlib import Path

WORKERS = 16
SESSION_HOURS = 10
RESUME_DB = ""  # Optional path to a state.sqlite3 mounted as a Kaggle input.
INPUT_BUNDLE = None  # Imported copies can set MALWEAVE_RANDS_INPUTS as a JSON secret.

STAGING = Path("/kaggle/temp/malweave-rands-audit")
OUTPUT = Path("/kaggle/working/rands-audit")
REGION = "ap-southeast-1"
PYTHON = "3.12.12"
BOTO3 = "1.43.123"


In [ ]:
import hashlib
import json
import shutil
import subprocess
import tarfile
from urllib.request import urlopen

from kaggle_secrets import UserSecretsClient

os.umask(0o077)
STAGING.mkdir(parents=True, exist_ok=True)
secrets = UserSecretsClient()
bundle = INPUT_BUNDLE or json.loads(secrets.get_secret("MALWEAVE_RANDS_INPUTS"))
env = dict(
    os.environ,
    AWS_DEFAULT_REGION=REGION,
    MALWEAVE_S3_ACCOUNT_ID=secrets.get_secret("MALWEAVE_S3_ACCOUNT_ID"),
    AWS_EC2_METADATA_DISABLED="true",
    UV_CACHE_DIR=str(STAGING / "uv-cache"),
    UV_PYTHON_INSTALL_DIR=str(STAGING / "python"),
    UV_PROJECT_ENVIRONMENT=str(STAGING / "env"),
    PYTHONUNBUFFERED="1",
)
env.pop("VIRTUAL_ENV", None)
for name in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    env[name] = secrets.get_secret(name)
try:
    env["AWS_SESSION_TOKEN"] = secrets.get_secret("AWS_SESSION_TOKEN")
except Exception:
    env.pop("AWS_SESSION_TOKEN", None)

def sha256_file(path):
    with path.open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()

UV_SHA256 = "5c43f82077ff0cd5aec588286cbabd89913e4d045bd4e8aa60b20b3ecffc36e3"
UV = STAGING / "uv"
if not UV.is_file():
    archive = STAGING / "uv.tar.gz"
    with (
        urlopen("https://github.com/astral-sh/uv/releases/download/0.11.9/"
                "uv-x86_64-unknown-linux-gnu.tar.gz", timeout=120) as source,
        archive.open("wb") as target,
    ):
        shutil.copyfileobj(source, target)
    if sha256_file(archive) != UV_SHA256:
        raise RuntimeError("Downloaded uv archive failed SHA-256 verification.")
    with tarfile.open(archive, "r:gz") as source:
        member = source.getmember("uv-x86_64-unknown-linux-gnu/uv")
        with source.extractfile(member) as binary, UV.open("wb") as target:
            shutil.copyfileobj(binary, target)
    UV.chmod(0o700)
    archive.unlink()

subprocess.run([str(UV), "python", "install", PYTHON], env=env, cwd=STAGING, check=True)
input_archive = STAGING / "audit-inputs.tar.gz"
if not input_archive.is_file() or sha256_file(input_archive) != bundle["sha256"]:
    temporary = input_archive.with_suffix(".download")
    temporary.unlink(missing_ok=True)
    download = """
import sys
import boto3
boto3.client("s3").download_file(*sys.argv[1:4])
"""
    subprocess.run(
        [str(UV), "run", "--no-project", "--python", PYTHON, "--with", f"boto3=={BOTO3}",
         "python", "-c", download, bundle["bucket"], bundle["key"], str(temporary)],
        env=env, cwd=STAGING, check=True,
    )
    if temporary.stat().st_size != bundle["bytes"] or sha256_file(temporary) != bundle["sha256"]:
        raise RuntimeError("RanDS input bundle failed size or SHA-256 verification.")
    temporary.replace(input_archive)

with tarfile.open(input_archive, "r:gz") as source:
    members = source.getmembers()
    if any(not (member.isfile() or member.isdir()) for member in members):
        raise RuntimeError("Expected only regular files and directories in the RanDS source bundle.")
    source.extractall(STAGING, members=members, filter="data")

REPO = STAGING / "repo"
print("Verified RanDS source bundle. The audit runner reads the published S3 inventory.", flush=True)


In [ ]:
command = [
    str(UV), "run", "--locked", "--python", PYTHON, "--no-default-groups",
    "--with", f"boto3=={BOTO3}", "python", "scripts/run-dataset-audit-kaggle.py", "full",
    "--dataset", DATASET, "--job", str(STAGING / "job"), "--output", str(OUTPUT),
    "--workers", str(WORKERS), "--session-seconds", str(SESSION_HOURS * 3600),
]
if LIMIT is not None:
    command.extend(["--limit", str(LIMIT)])

if RESUME_DB:
    checkpoint = Path(RESUME_DB)
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
else:
    attached = sorted(
        path for path in Path("/kaggle/input").rglob(f"{DATASET}-audit/state.sqlite3")
        if path.is_file()
    )
    if len(attached) > 1:
        raise RuntimeError("Set RESUME_DB to the checkpoint you want to resume.")
    checkpoint = attached[0] if attached else STAGING / "resume/state.sqlite3"

if checkpoint.is_file():
    command.extend(["--restore", str(checkpoint)])

subprocess.run(command, env=env, cwd=REPO, check=True)
